<a href="https://colab.research.google.com/github/OdysseusPolymetis/atelier_humanistica2026/blob/main/4_basic_topic_modeling_bertopic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Topic modeling avec BERTopic pour corpus grecs et latins

Objectif du notebook :

1. charger un corpus grec ou latin ;
2. préparer les textes pour le topic modeling ;
3. calculer des embeddings adaptés autant que possible aux langues anciennes ;
4. entraîner un modèle BERTopic ;
5. interpréter et exporter les topics.

**Idée importante :** BERTopic peut être utilisé sur du grec ancien ou du latin, mais il n'est pas "nativement philologique".  
Il faut être attentif à trois points :

- la qualité des embeddings ;
- la taille des segments ;
- la représentation lexicale des topics, très sensible aux formes fléchies.


## 0. Installation

Dans Colab, exécuter cette cellule une seule fois.

Le paquet `bertopic` installe les briques principales : UMAP, HDBSCAN, c-TF-IDF.  
`sentence-transformers` sert à calculer les embeddings.

In [ ]:
!pip install -q bertopic sentence-transformers pandas numpy scikit-learn regex plotly beautifulsoup4 requests tqdm

## 1. Imports et paramètres généraux

## Source du corpus
* Option A : charger un CSV local.
Format attendu : au moins une colonne 'text'.
* Option B : construire automatiquement un corpus depuis Scaife ATLAS.
Le téléchargement est mis en cache dans SCAIFE_CACHE_PATH pour éviter d'interroger le serveur à chaque exécution.
*

In [ ]:
import os
import re
import time
import unicodedata
import warnings
from urllib.parse import quote, unquote, urljoin

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

DATA_PATH = None  # ex. "/content/mon_corpus.csv"

USE_SCAIFE_ATLAS = True
SCAIFE_CACHE_PATH = "scaife_atlas_corpus.csv"

# Ici le paramètre du nombre maximal de passages récupérés par édition.

SCAIFE_MAX_PASSAGES_PER_TEXT = 40

# Pause entre les requêtes : ça ralentit mais ça peut être utile si le serveur est limité.
SCAIFE_SLEEP = 0.15

SCAIFE_EDITIONS = [
    {
        "edition_urn": "urn:cts:latinLit:phi0690.phi003.perseus-lat2",
        "author": "Vergilius",
        "title": "Aeneis",
        "language": "lat"
    },
    {
        "edition_urn": "urn:cts:latinLit:phi0448.phi001.perseus-lat2",
        "author": "Caesar",
        "title": "De Bello Gallico",
        "language": "lat"
    },
    {
        "edition_urn": "urn:cts:latinLit:phi0474.phi052.perseus-lat2",
        "author": "Cicero",
        "title": "De Amicitia",
        "language": "lat"
    },
    {
        "edition_urn": "urn:cts:latinLit:phi0474.phi048.perseus-lat1",
        "author": "Cicero",
        "title": "De Finibus Bonorum et Malorum",
        "language": "lat"
    },
    {
        "edition_urn": "urn:cts:latinLit:phi0474.phi016.perseus-lat2",
        "author": "Cicero",
        "title": "Pro Archia Poeta",
        "language": "lat"
    },
]

# Pour tester en grec, remplacer SCAIFE_EDITIONS par exemple par :
GREEK_EXAMPLE_EDITIONS = [
    {
        "edition_urn": "urn:cts:greekLit:tlg0012.tlg001.perseus-grc2",
        "author": "Homerus",
        "title": "Ilias",
        "language": "grc"
    },
    {
        "edition_urn": "urn:cts:greekLit:tlg0012.tlg002.perseus-grc2",
        "author": "Homerus",
        "title": "Odyssea",
        "language": "grc"
    },
    {
        "edition_urn": "urn:cts:greekLit:tlg0003.tlg001.perseus-grc2",
        "author": "Thucydides",
        "title": "Historiae",
        "language": "grc"
    },
    {
        "edition_urn": "urn:cts:greekLit:tlg0059.tlg030.perseus-grc2",
        "author": "Plato",
        "title": "Respublica",
        "language": "grc"
    },
]

CLASSICS_EMBEDDING_MODEL = "bowphs/SPhilBerta"
FALLBACK_EMBEDDING_MODEL = "paraphrase-multilingual-MiniLM-L12-v2" # celui-là c'est optionnel, si jamais le premier ne marche pas.

## 2. Corpus de démonstration

Le corpus ci-dessous est volontairement petit.  
Il sert seulement à vérifier que le pipeline fonctionne.

Pour un vrai topic modeling, il faut davantage de documents ou de segments : idéalement plusieurs centaines de passages.

In [ ]:
demo_data = [
    {"id": "lat_001", "language": "lat", "author": "Vergilius", "title": "Aeneis", "text": "Arma virumque cano, Troiae qui primus ab oris Italiam fato profugus Laviniaque venit litora."},
    {"id": "lat_002", "language": "lat", "author": "Vergilius", "title": "Aeneis", "text": "Multum ille et terris iactatus et alto vi superum saevae memorem Iunonis ob iram."},
    {"id": "lat_003", "language": "lat", "author": "Vergilius", "title": "Aeneis", "text": "Musa, mihi causas memora, quo numine laeso quidve dolens regina deum tot volvere casus."},
    {"id": "lat_004", "language": "lat", "author": "Cicero", "title": "Catilinariae", "text": "Quousque tandem abutere, Catilina, patientia nostra? quam diu etiam furor iste tuus nos eludet?"},
    {"id": "lat_005", "language": "lat", "author": "Cicero", "title": "Catilinariae", "text": "O tempora, o mores! Senatus haec intellegit, consul videt; hic tamen vivit."},
    {"id": "lat_006", "language": "lat", "author": "Seneca", "title": "Epistulae", "text": "Non scholae sed vitae discimus, et animus cotidie exercendus est."},
    {"id": "lat_007", "language": "lat", "author": "Seneca", "title": "Epistulae", "text": "Ira est cupiditas ulciscendae iniuriae, quae saepe rationem perturbat."},
    {"id": "lat_008", "language": "lat", "author": "Ovidius", "title": "Metamorphoses", "text": "In nova fert animus mutatas dicere formas corpora; di, coeptis aspirate meis."},

    {"id": "grc_001", "language": "grc", "author": "Homerus", "title": "Ilias", "text": "Μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος οὐλομένην."},
    {"id": "grc_002", "language": "grc", "author": "Homerus", "title": "Ilias", "text": "πολλὰς δ᾽ ἰφθίμους ψυχὰς Ἄϊδι προΐαψεν ἡρώων."},
    {"id": "grc_003", "language": "grc", "author": "Homerus", "title": "Odyssea", "text": "Ἄνδρα μοι ἔννεπε, Μοῦσα, πολύτροπον, ὃς μάλα πολλὰ πλάγχθη."},
    {"id": "grc_004", "language": "grc", "author": "Plato", "title": "Respublica", "text": "δικαιοσύνην ζητοῦμεν, τί ποτ᾽ ἐστίν, καὶ ἐν πόλει καὶ ἐν ψυχῇ."},
    {"id": "grc_005", "language": "grc", "author": "Plato", "title": "Respublica", "text": "ἡ ψυχὴ τρία μέρη ἔχει, λογιστικόν, θυμοειδές, ἐπιθυμητικόν."},
    {"id": "grc_006", "language": "grc", "author": "Sophocles", "title": "Antigone", "text": "πολλὰ τὰ δεινὰ κοὐδὲν ἀνθρώπου δεινότερον πέλει."},
    {"id": "grc_007", "language": "grc", "author": "Euripides", "title": "Medea", "text": "θυμὸς δὲ κρείσσων τῶν ἐμῶν βουλευμάτων γίνεται."},
    {"id": "grc_008", "language": "grc", "author": "Aristoteles", "title": "Ethica", "text": "πᾶσα τέχνη καὶ πᾶσα μέθοδος ἀγαθοῦ τινὸς ἐφίεσθαι δοκεῖ."},
]

df_demo = pd.DataFrame(demo_data)
df_demo

## 2bis. Construire un corpus depuis Scaife ATLAS

Cette section récupère des passages directement depuis **Scaife ATLAS**.

Principe :

1. on part d'une liste d'URNs d'éditions ;
2. pour chaque édition, on repère le premier passage ;
3. on suit le lien `next` pour récupérer les passages suivants ;
4. on sauvegarde le tout en CSV pour éviter de refaire les requêtes.

Les unités récupérées par ATLAS sont déjà assez adaptées à BERTopic : environ 30 vers pour l'épopée, ou quelques sections pour la prose.

In [ ]:
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

ATLAS_BASE = "https://atlas.perseus.tufts.edu"

def atlas_library_url(urn: str) -> str:
    return urljoin(ATLAS_BASE, f"/library/{quote(urn, safe='')}/")

def atlas_passage_url(passage_urn: str, fmt: str | None = None) -> str:
    path = f"/library/passage/{quote(passage_urn, safe='')}/"
    if fmt:
        path += f"{fmt}/"
    return urljoin(ATLAS_BASE, path)

def clean_downloaded_text(text: str) -> str:
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def split_passage_urn(passage_urn: str):
    edition_urn, reference = passage_urn.rsplit(":", 1)
    return edition_urn, reference

def get_first_passage_urn(edition_urn: str) -> str:
    r = requests.get(atlas_library_url(edition_urn), timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    for a in soup.find_all("a", href=True):
        if "/library/passage/" in a["href"]:
            href = urljoin(ATLAS_BASE, a["href"])
            m = re.search(r"/library/passage/([^/]+)/", href)
            if m:
                return unquote(m.group(1))
    raise ValueError(f"Impossible de trouver le premier passage pour {edition_urn}")

def get_next_passage_urn(page_html: str) -> str | None:
    soup = BeautifulSoup(page_html, "html.parser")
    for a in soup.find_all("a", href=True):
        if a.get_text(strip=True).lower() == "next" and "/library/passage/" in a["href"]:
            href = urljoin(ATLAS_BASE, a["href"])
            m = re.search(r"/library/passage/([^/]+)/", href)
            if m:
                return unquote(m.group(1))
    return None

def fetch_passage_text(passage_urn: str) -> str:
    r = requests.get(atlas_passage_url(passage_urn, fmt="text"), timeout=30)
    r.raise_for_status()
    return clean_downloaded_text(r.text)

def fetch_passage_page(passage_urn: str) -> str:
    r = requests.get(atlas_passage_url(passage_urn), timeout=30)
    r.raise_for_status()
    return r.text

def fetch_atlas_edition(edition_info: dict, max_passages: int = 40, sleep: float = 0.15) -> pd.DataFrame:
    first = get_first_passage_urn(edition_info["edition_urn"])
    current = first
    rows = []

    for _ in tqdm(range(max_passages), desc=f"{edition_info.get('author', '')} — {edition_info.get('title', '')}"):
        page_html = fetch_passage_page(current)
        text = fetch_passage_text(current)
        edition_urn, reference = split_passage_urn(current)

        rows.append({
            "id": current,
            "edition_urn": edition_urn,
            "reference": reference,
            "author": edition_info.get("author", ""),
            "title": edition_info.get("title", ""),
            "language": edition_info.get("language", ""),
            "text": text
        })

        nxt = get_next_passage_urn(page_html)
        if not nxt:
            break
        current = nxt
        time.sleep(sleep)

    return pd.DataFrame(rows)

def build_scaife_corpus(editions: list[dict], max_passages_per_text: int = 40, sleep: float = 0.15) -> pd.DataFrame:
    parts = []
    for edition in editions:
        try:
            part = fetch_atlas_edition(
                edition,
                max_passages=max_passages_per_text,
                sleep=sleep
            )
            parts.append(part)
        except Exception as e:
            print(f"Échec pour {edition.get('edition_urn')}: {e}")
    if not parts:
        raise RuntimeError("Aucun texte n'a pu être récupéré.")
    return pd.concat(parts, ignore_index=True)

# Décommenter pour utiliser le corpus grec au lieu du corpus latin :
# SCAIFE_EDITIONS = GREEK_EXAMPLE_EDITIONS

## 3. Charger le corpus

Priorité :

1. si `DATA_PATH` pointe vers un CSV local, on l'utilise ;
2. sinon, si `USE_SCAIFE_ATLAS = True`, on construit ou recharge un corpus depuis Scaife ATLAS ;
3. sinon, on utilise le mini-corpus de démonstration.

In [ ]:
if DATA_PATH and os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"Corpus chargé depuis CSV local : {DATA_PATH}")

elif USE_SCAIFE_ATLAS:
    if os.path.exists(SCAIFE_CACHE_PATH):
        df = pd.read_csv(SCAIFE_CACHE_PATH)
        print(f"Corpus Scaife ATLAS rechargé depuis le cache : {SCAIFE_CACHE_PATH}")
    else:
        df = build_scaife_corpus(
            SCAIFE_EDITIONS,
            max_passages_per_text=SCAIFE_MAX_PASSAGES_PER_TEXT,
            sleep=SCAIFE_SLEEP
        )
        df.to_csv(SCAIFE_CACHE_PATH, index=False)
        print(f"Corpus Scaife ATLAS sauvegardé dans : {SCAIFE_CACHE_PATH}")

else:
    df = df_demo.copy()
    print("Aucun corpus externe trouvé : utilisation du corpus de démonstration.")

assert "text" in df.columns, "Le corpus doit contenir une colonne 'text'."

if "id" not in df.columns:
    df["id"] = [f"doc_{i:04d}" for i in range(len(df))]

for col in ["author", "title", "language", "reference"]:
    if col not in df.columns:
        df[col] = ""

df["text"] = df["text"].fillna("").astype(str)
df = df[df["text"].str.strip().astype(bool)].reset_index(drop=True)

print(df.shape)
display(df[["id", "author", "title", "language", "reference", "text"]].head())
display(df.groupby(["language", "author", "title"]).size().reset_index(name="n_passages"))

## 4. Normalisation légère

Pour les langues anciennes, on peut hésiter entre :

- conserver les formes originales pour l'embedding ;
- normaliser ou lemmatiser pour les mots représentatifs des topics. Idéalement il faudrait vraiment lemmatiser au moins. Mais nous n'aurons pas le temps.

Ici, on conserve `text` pour les embeddings, et on crée `text_for_topics` pour la représentation c-TF-IDF.

In [ ]:
LATIN_STOPWORDS = {
    "et", "in", "de", "ad", "non", "est", "sunt", "cum", "ut", "quae", "qui", "quo",
    "nam", "sed", "aut", "nec", "per", "ab", "ex", "a", "o", "si", "se", "me", "te",
    "hoc", "haec", "hic", "ille", "illa", "id", "eum", "eam", "enim", "que"
}

GREEK_STOPWORDS = {
    "καὶ", "δὲ", "γὰρ", "μὲν", "τε", "τὸ", "τὰ", "τῆς", "τῶν", "τῷ", "τοῦ",
    "ὁ", "ἡ", "οἱ", "αἱ", "ἐν", "εἰς", "ἐκ", "οὐ", "οὐκ", "μὴ", "ὡς", "τις",
    "τι", "δέ", "μέν", "γαρ", "και"
}

def strip_accents(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn"
    )

def normalize_for_topics(text: str, remove_diacritics: bool = False) -> str:
    text = text.lower()
    if remove_diacritics:
        text = strip_accents(text)
    text = re.sub(r"[^a-zA-ZΑ-Ωα-ωἀ-῾]+", " ", text)
    tokens = [tok for tok in text.split() if len(tok) > 2]
    tokens = [tok for tok in tokens if tok not in LATIN_STOPWORDS and tok not in GREEK_STOPWORDS]
    return " ".join(tokens)

df["text_for_topics"] = df["text"].apply(lambda x: normalize_for_topics(x, remove_diacritics=False))

df[["id", "text", "text_for_topics"]].head()

## 5. Calculer les embeddings

Pour grec ancien / latin, on essaie d'abord `bowphs/SPhilBerta`, pensé pour la philologie classique.  
Si le chargement échoue, on bascule vers un modèle multilingue généraliste.

In [ ]:
from sentence_transformers import SentenceTransformer

def load_sentence_model(primary_model: str, fallback_model: str):
    try:
        print(f"Tentative de chargement : {primary_model}")
        model = SentenceTransformer(primary_model)
        print(f"Modèle chargé : {primary_model}")
        return model, primary_model
    except Exception as e:
        print(f"Échec du chargement de {primary_model}: {e}")
        print(f"Bascule vers : {fallback_model}")
        model = SentenceTransformer(fallback_model)
        return model, fallback_model

sentence_model, embedding_model_name = load_sentence_model(CLASSICS_EMBEDDING_MODEL, FALLBACK_EMBEDDING_MODEL)

raw_docs = df["text"].tolist()
topic_docs = df["text_for_topics"].tolist()

embeddings = sentence_model.encode(
    raw_docs,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings.shape

## 6. Entraîner BERTopic

Paramètres importants :

- `min_topic_size` : augmentez-le si le corpus est grand ;
- `n_neighbors` dans UMAP : plus faible pour petits corpus, plus élevé pour corpus substantiels ;
- `ngram_range` : `(1, 2)` aide souvent à faire émerger des expressions.

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

n_docs = len(topic_docs)
n_neighbors = min(15, max(2, n_docs - 1))
min_topic_size = 2 if n_docs < 50 else 5

vectorizer_model = CountVectorizer(
    tokenizer=lambda x: x.split(),
    token_pattern=None,
    lowercase=False,
    min_df=1,
    max_df=0.95,
    ngram_range=(1, 2)
)

umap_model = UMAP(
    n_neighbors=n_neighbors,
    n_components=min(5, max(2, n_docs - 2)),
    min_dist=0.0,
    metric="cosine",
    random_state=RANDOM_STATE
)

hdbscan_model = HDBSCAN(
    min_cluster_size=min_topic_size,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

topic_model = BERTopic(
    embedding_model=None,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(topic_docs, embeddings=embeddings)

df["topic"] = topics
topic_info = topic_model.get_topic_info()
topic_info

## 7. Explorer les topics

In [ ]:
for topic_id in topic_info["Topic"].tolist():
    if topic_id == -1:
        continue
    print("=" * 80)
    print(f"Topic {topic_id}")
    print(topic_model.get_topic(topic_id))
    print()
    display(df.loc[df["topic"] == topic_id, ["id", "author", "title", "language", "text"]].head(10))

## 8. Visualisations

Dans Colab, les figures Plotly s'affichent directement.

In [ ]:
topic_model.visualize_barchart(top_n_topics=10)

In [ ]:
topic_model.visualize_documents(topic_docs, embeddings=embeddings)

## 9. Exporter les résultats

In [ ]:
df.to_csv("bertopic_documents_with_topics.csv", index=False)
topic_info.to_csv("bertopic_topic_info.csv", index=False)

print("Fichiers exportés :")
print("- bertopic_documents_with_topics.csv")
print("- bertopic_topic_info.csv")

## Questions à poser :

- Les topics sont-ils philologiquement interprétables ?
- Les mots représentatifs sont-ils trop dépendants de la flexion ?
- Le modèle regroupe-t-il des passages par thème, par auteur, par genre ou par langue ?
- Que changerait une vraie lemmatisation ?